# Part 2. Structured Outputs

In [58]:
import json
import textwrap
import instructor
from datetime import date
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Annotated, Literal


In [5]:
# Let's redefine and slightly modify the response generation function 
# from the part 1 of our tutorial.

# Define a wrapper function that will send requests to an LLM and receive responses
def generate_response(
        client: OpenAI | instructor.Instructor,
        model: str,
        user_prompt: str,
        system_prompt: str = "You are a helpful assistant.",
        temperature: float = 0.5       
) -> str:
    """Sends a request to an LLM and and returns a response."""
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

#### 1. Motivational Example

In [12]:
# Create an OpenAI client
openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [ ]:
role = "a Python Developer"
task_description = "to develop Python code based on the specification provided"
context = "write a Python function that generates Fibonacci sequence; use a `while` loop;"

output_format = """
Generate a brief explanation of your solution and a block of Python code.
Return ONLY a valid JSON object matching the following schema:

{
    "explanation": string,
    "code": string
}

- Do NOT add text outside the JSON object.
"""

# DO NOT ADD ANY TEXT OUTSIDE of the JSON object.

In [45]:
system_prompt = f"You are {role}. Your task is {task_description}."
user_prompt = f"""
Here are additional instructions: {context}.
OUTPUT FORMAT: {output_format}
"""


print("SYSTEM PROMPT:", system_prompt, sep="\n")
print("-" * 100)
print("USER PROMPT:", user_prompt, sep="\n")

SYSTEM PROMPT:
You are a Python Developer. Your task is to develop Python code based on the specification provided.
----------------------------------------------------------------------------------------------------
USER PROMPT:

Here are additional instructions: write a Python function that generates Fibonacci sequence; use a `while` loop;.
OUTPUT FORMAT: 
Generate a brief explanation of your solution and a block of Python code.
Return ONLY a valid JSON object matching the following schema:

{
    "explanation": string,
    "code": string
}

- Do NOT add text outside the JSON object.




In [46]:
response = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=user_prompt,
    system_prompt=system_prompt,
)

print(response)

{
    "explanation": "This function generates the Fibonacci sequence up to the nth number. It uses a while loop to iterate over the sequence.",
    "code": "def fibonacci(n):\n    fib_sequence = [0, 1]\n    while len(fib_sequence) < n:\n        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])\n    return fib_sequence"
}


In [49]:
# parse the response with json module
converted_response = json.loads(response)
print(f"Type: {type(converted_response)}")

Type: <class 'dict'>


In [55]:
# That's a Python dictionary! 
print(converted_response['explanation'])
print("-" * 100)
print(converted_response['code'])

This function generates the Fibonacci sequence up to the nth number. It uses a while loop to iterate over the sequence.
----------------------------------------------------------------------------------------------------
def fibonacci(n):
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence


#### 2. Structured Outputs with Instructor

In [56]:
# Wrap the OpenaAI client into the Instructor client
instructor_client = instructor.from_openai(
    openai_client,
    mode=instructor.Mode.JSON # When using with ollama, provide mode parameter 
)

In [75]:
# Create a pydantic model

class User(BaseModel):
    first_name: str = Field(..., description="user's first name")
    second_name: str = Field(..., description="user's second name")
    date_of_birth: date = Field(..., description="user's date of birth")
    address_number: str = Field(..., description="building number where a user lives")
    street_name: str = Field(..., description="street where user lives")
    city_name: str = Field(..., description="city in which user lives")
    state: str = Field(..., description="state (within) the United States of America where user lives")
    zip_code: str = Field(..., description="a zip code", min_length=5, max_length=5)
    credit_score: int = Field(..., description="a credit score of a given user in the range between 300 and 350", ge=300, le=350)
    sex: Annotated[Literal["Male", "Female", "Undefined"], Field(description="user's sex")]

In [76]:
# Let's see the description of the pydantic schema for User 
User.model_fields

{'first_name': FieldInfo(annotation=str, required=True, description="user's first name"),
 'second_name': FieldInfo(annotation=str, required=True, description="user's second name"),
 'date_of_birth': FieldInfo(annotation=date, required=True, description="user's date of birth"),
 'address_number': FieldInfo(annotation=str, required=True, description='building number where a user lives'),
 'street_name': FieldInfo(annotation=str, required=True, description='street where user lives'),
 'city_name': FieldInfo(annotation=str, required=True, description='city in which user lives'),
 'state': FieldInfo(annotation=str, required=True, description='state (within) the United States of America where user lives'),
 'zip_code': FieldInfo(annotation=str, required=True, description='a zip code', metadata=[MinLen(min_length=5), MaxLen(max_length=5)]),
 'credit_score': FieldInfo(annotation=int, required=True, description='a credit score of a given user in the range between 300 and 350', metadata=[Ge(ge=

In [77]:
system_prompt = "You are a data generation system. Your task is to create content based on the specifications provided by the user."
user_prompt = "Create a random user's data matching the data schema provided."

In [78]:
generated_user_data = instructor_client.chat.completions.create(
    model="llama3.2:3b",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_model=User,
    max_retries=3
)

print("Type: ", type(generated_user_data))
print(generated_user_data)

Type:  <class '__main__.User'>
first_name='Emily' second_name='Doe' date_of_birth=datetime.date(1992, 10, 12) address_number='123' street_name='Main St' city_name='Arlington' state='Virginia' zip_code='22201' credit_score=325 sex='Female'


In [79]:
print(generated_user_data.first_name)

Emily


In [72]:
generated_user_data_json = generated_user_data.model_dump_json(indent=2)
print(generated_user_data_json)
print("Type:", type(generated_user_data_json))

{
  "first_name": "Julian",
  "second_name": "Rodriguez",
  "date_of_birth": "1999-04-18",
  "address_number": "1234",
  "street_name": "Franklin Ave",
  "city_name": "Austin",
  "state": "Texas",
  "zip_code": "51285",
  "credit_score": 328,
  "sex": "Male"
}
Type: <class 'str'>


In [86]:
# Let's generate more users

users_list = []

for i in range(20):
    print(f"Iteration # {i+1}")
    
    try:
        generated_user = instructor_client.chat.completions.create(
            model="llama3.2:3b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_model=User,
            max_retries=2,
            temperature=1.0
        )
        
        users_list.append(generated_user)
    
    except Exception as ex:
        print(f"Something went wrong at iteration # {i+1}")
        print(ex)

Iteration # 1
Iteration # 2
Iteration # 3
Something went wrong at iteration # 3
<failed_attempts>

<generation number="1">
<exception>
    10 validation errors for User
first_name
  Field required [type=missing, input_value={'properties': {'first_na...User', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
second_name
  Field required [type=missing, input_value={'properties': {'first_na...User', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
date_of_birth
  Field required [type=missing, input_value={'properties': {'first_na...User', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
address_number
  Field required [type=missing, input_value={'properties': {'first_na...User', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
street_name

In [87]:
users_list

[User(first_name='John', second_name='Doe', date_of_birth=datetime.date(1990, 9, 21), address_number='123', street_name='Main Street', city_name='Anytown', state='California', zip_code='92311', credit_score=325, sex='Male'),
 User(first_name='Jasmine', second_name='Brown', date_of_birth=datetime.date(1997, 2, 17), address_number='342', street_name='Broadway', city_name='Chicago', state='IL', zip_code='94105', credit_score=325, sex='Female'),
 User(first_name='John', second_name='Doe', date_of_birth=datetime.date(1990, 1, 1), address_number='123', street_name='Main ST', city_name='New York', state='NY', zip_code='10001', credit_score=320, sex='Male'),
 User(first_name='Emily', second_name='Harris', date_of_birth=datetime.date(1995, 3, 15), address_number='123', street_name='Main Street', city_name='New York', state='NY', zip_code='10001', credit_score=325, sex='Female'),
 User(first_name='Emily', second_name='Doe', date_of_birth=datetime.date(1992, 4, 22), address_number='123', street_n

In [90]:
# Too many records with John Doe, let's make an example of him.
print(users_list[0].model_dump_json(indent=2))

{
  "first_name": "John",
  "second_name": "Doe",
  "date_of_birth": "1990-09-21",
  "address_number": "123",
  "street_name": "Main Street",
  "city_name": "Anytown",
  "state": "California",
  "zip_code": "92311",
  "credit_score": 325,
  "sex": "Male"
}


In [ ]:
# Let's include an example (One-Shot Learning)

system_prompt = "You are a data generation system. Your task is to create content based on the specifications provided by the user."
user_prompt = """
Create a random user's data matching the data schema provided.

OUTPUT EXAMPLE:

{
  "first_name": "John",
  "second_name": "Doe",
  "date_of_birth": "1990-09-21",
  "address_number": "123",
  "street_name": "Main Street",
  "city_name": "New York",
  "state": "New York",
  "zip_code": "10001",
  "credit_score": 325,
  "sex": "Male"
}

Do NOT repeat the data from the EXAMPLE.
"""

# Do NOT repeat the data from the EXAMPLE.

In [98]:
# Let's generate more users

users_list = []

for i in range(20):
    print(f"Iteration # {i+1}")
    
    try:
        generated_user = instructor_client.chat.completions.create(
            model="llama3.2:3b",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_model=User,
            max_retries=2,
            temperature=1.0
        )
        
        users_list.append(generated_user)
    
    except Exception as ex:
        print(f"Something went wrong at iteration # {i+1}")
        print(ex)

Iteration # 1
Iteration # 2
Iteration # 3
Iteration # 4
Iteration # 5
Iteration # 6
Iteration # 7
Iteration # 8
Iteration # 9
Iteration # 10
Iteration # 11
Iteration # 12
Iteration # 13
Iteration # 14
Iteration # 15
Iteration # 16
Iteration # 17
Iteration # 18
Iteration # 19
Iteration # 20


In [99]:
users_list

[User(first_name='Oliver', second_name='White', date_of_birth=datetime.date(1992, 8, 14), address_number='456', street_name='Park Avenue', city_name='San Francisco', state='California', zip_code='94101', credit_score=332, sex='Female'),
 User(first_name='Amanda', second_name='White', date_of_birth=datetime.date(1994, 5, 22), address_number='456', street_name='Broadway', city_name='Los Angeles', state='California', zip_code='90001', credit_score=338, sex='Female'),
 User(first_name='Elijah', second_name='Wright', date_of_birth=datetime.date(1985, 4, 12), address_number='456', street_name='Park Avenue', city_name='Los Angeles', state='California', zip_code='90210', credit_score=320, sex='Female'),
 User(first_name='Alexander', second_name='Garcia', date_of_birth=datetime.date(1985, 2, 28), address_number='456', street_name='Oak Street', city_name='Los Angeles', state='', zip_code='90007', credit_score=320, sex='Male'),
 User(first_name='Samantha', second_name='Lopez', date_of_birth=datet